In [2]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
df = pd.read_csv('student_data.csv')
df

,Studying Hours,Theoretical classes attended (%),Repeating Student,Continuous Assessment,Exam (0/20)
0,8.7,90,False,14,10
1,11.7,68,False,12,9
2,18.2,30,True,11,8
3,10.5,41,False,10,9
4,6.5,10,False,12,5
5,9.0,35,True,11,14
6,10.2,85,False,13,11
7,7.3,20,True,13,13
8,15.5,48,True,14,15
9,12.0,80,False,15,11


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
 
# Convert True/False to 1/0
df['Repeating Student'] = df['Repeating Student'].map(
    {True: 1, False: 0, 'True': 1, 'False': 0}
)
 
FEATURES = [
    'Studying Hours',
    'Theoretical classes attended (%)',
    'Repeating Student',
    'Continuous Assessment',
]
 
X = df[FEATURES].values
y = df['Exam (0/20)'].values
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
model = LinearRegression()
model.fit(X_scaled, y)
 
print("Model trained successfully!")

Model trained successfully!


In [10]:
# Sliders
style  = {'description_width': '230px'}
layout = widgets.Layout(width='460px')
 
w_hours = widgets.FloatSlider(
    value=10.0, min=0.0, max=30.0, step=0.5,
    description='Studying Hours per week:',
    style=style, layout=layout, readout_format='.1f',
)
w_theory = widgets.IntSlider(
    value=75, min=0, max=100, step=5,
    description='Theoretical Classes Attended (%):',
    style=style, layout=layout,
)
w_repeat = widgets.ToggleButtons(
    options=[('First time student', 0), ('Repeating student', 1)],
    description='Student type:',
    style={'description_width': '180px', 'button_width': '140px'},
)
w_ca = widgets.FloatSlider(
    value=12.0, min=0.0, max=20.0, step=0.5,
    description='Continuous Assessment grade:',
    style=style, layout=layout, readout_format='.1f',
)
 
btn = widgets.Button(
    description='Predict grade',
    button_style='primary',
    layout=widgets.Layout(width='160px', height='36px', margin='12px 0 0 0'),
)
out = widgets.Output()
 
def on_predict(b):
    with out:
        clear_output(wait=True)
 
        # Build the input and predict
        profile = np.array([[w_hours.value, w_theory.value, w_repeat.value, w_ca.value]])
        profile_scaled = scaler.transform(profile)
        predicted = float(model.predict(profile_scaled)[0])
        predicted = round(max(0.0, min(20.0, predicted)), 1)
 
        # Grade label
        if predicted >= 18:
            label, color = 'Excellent', '#2ecc71'
        elif predicted >= 14:
            label, color = 'Good', '#27ae60'
        elif predicted >= 9.5:
            label, color = 'Pass', '#f39c12'
        else:
            label, color = 'Fail', '#e74c3c'
 
        print(f"Predicted exam grade: {predicted} / 20  ({label})")
 
        # Grade bar
        fig, ax = plt.subplots(figsize=(7, 1.1))
        ax.barh([0], [20], color='#ecf0f1', height=0.5, edgecolor='#ccc')
        ax.barh([0], [predicted], color=color, height=0.5, edgecolor='none')
        ax.axvline(10, color='#aaa', linestyle='--', linewidth=0.8)
        ax.text(10.15, 0.32, 'Pass threshold (10)', fontsize=8, color='#888')
        ax.set_xlim(0, 20)
        ax.set_yticks([])
        ax.set_xlabel('Grade (0–20)')
        ax.spines[['top', 'right', 'left']].set_visible(False)
        plt.tight_layout()
        plt.show()
 
btn.on_click(on_predict)
 
# Layout
header = widgets.VBox([
    widgets.Label('🎓 Grade Predictor', style={'font_weight': 'bold', 'font_size': '16px'}),
    widgets.Label('Adjust the inputs and click Predict grade.'),
])
 
display(widgets.VBox(
    [header, w_hours, w_theory, w_repeat, w_ca, btn, out],
    layout=widgets.Layout(padding='16px', border='1px solid #ddd', width='500px')
))
